# Liquidation Signal — Benchmark

Heuristic and ML experiments with turnover sweep.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("..").resolve()))

In [ ]:
from strategies.liq_filter.train import run_experiment

DATA_DIR = "data"

# --- Heuristic benchmark ---
fitted, reports, df = run_experiment(
    DATA_DIR,
    name="heuristic",
    use_ml=False,
    symbols=("btcusdt", "ethusdt"),
    target_turnover_per_day=5e5,
    verbose=True,
)
display(df)

In [ ]:
# --- ML benchmark ---
fitted_ml, reports_ml, df_ml = run_experiment(
    DATA_DIR,
    name="lgbm",
    use_ml=True,
    symbols=("btcusdt", "ethusdt"),
    target_turnover_per_day=5e5,
    model_params={
        "learning_rate": 0.03,
        "num_leaves": 63,
        "objective": "huber",
    },
    verbose=True,
)
display(df_ml)

In [ ]:
# --- Turnover sweep ---
import pandas as pd

results = []
for target in [5e5, 1e6, 2e6, 5e6, 1e7, 2e7, 5e7]:
    _, _, sweep_df = run_experiment(
        DATA_DIR,
        name=f"sweep_{target:.0e}",
        use_ml=False,
        symbols=("btcusdt", "ethusdt"),
        target_turnover_per_day=target,
        verbose=False,
    )
    sweep_df["target_turnover"] = target
    results.append(sweep_df)

sweep = pd.concat(results, ignore_index=True)
display(sweep)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, sym in zip(axes, ("btcusdt", "ethusdt")):
    for tau in [30, 120, 300]:
        sub = sweep[(sweep["symbol"] == sym) & (sweep["tau"] == tau)].sort_values("target_turnover")
        ax.plot(sub["kept_turnover_per_day"], sub["score"], marker="o", label=f"τ={tau}s")
    ax.set_xscale("log")
    ax.axhline(0, color="k", linewidth=0.5, linestyle="--")
    ax.set_xlabel("Kept Turnover / Day ($)")
    ax.set_ylabel("Score (bps)")
    ax.set_title(sym.upper())
    ax.legend()
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e6:.1f}M"))

plt.suptitle("Score vs. Kept Turnover (Heuristic)", fontsize=13)
plt.tight_layout()
plt.show()